# MirrorTopology Step 1 — Phase D-2：第 1 波 bank 生成（family 単位；v0.1）
登録 CRN 表・bank 仕様（正規再導出）・受入れ済み D-1 共分散（固定 SHA）・凍結 loader・登録環境 hard gate のもとで，1 family の評価 bank（4×10⁶ 行×3 role＋family reference，必須 subset は batch 0 で f32 併走）・fitting bank・W₂ position bank を生成し，`d2_bank_registry.json` と call inventory を残す。**family を 1 つずつ別 run で実行**（E1 ≈0.5 h，E2/E7/E8 各 ≈1.4 h の見込み；sandbox 計時からの外挿で保証ではない）。出力は Drive に保存（≈2.7 GB／family）；repository には registry のみ。中断時は同じ commit で `REUSE_ROOT` に前回の OUT を指定して再開（完成 dir は全再検証後に再利用）。label なし・較正なし。

In [ ]:
# --- 0. OUTER LAUNCHER LOCK (the only editable cell)
REPO_URL = 'https://github.com/tsujikeita/mirror-topology.git'
REPO_COMMIT = '<full 40-hex commit of the verification target>'
EXPECTED_INVENTORY_SHA256 = '<sha256 of engine/phaseB/B2_completion_inventory.json inside that commit>'
FAMILY = 'E1'                      # one of E1 / E2 / E7 / E8 ; run four separate notebooks
DRIVE_DIR = '/content/drive/MyDrive/MirrorTopology_D2'     # persistent output root
REUSE_ROOT = ''                    # '' for a fresh run; or the previous attempt's OUT/d2 (same commit) to reuse its completed directories
LAUNCHER_ID = 'MirrorTopology_Step1_D2_bankgen_v0.1'


In [ ]:
# --- 1. fresh scratch checkout at C; inventory bound; d/ pins+script bound; Phase C packet at the same commit
import subprocess, sys, os, json, hashlib, shutil, time, re
sha=lambda p: hashlib.sha256(open(p,'rb').read()).hexdigest()
assert re.fullmatch(r'[0-9a-f]{40}', REPO_COMMIT) and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_INVENTORY_SHA256) and FAMILY in ('E1','E2','E7','E8'), 'launcher lock not filled'
from google.colab import drive; drive.mount('/content/drive'); os.makedirs(DRIVE_DIR, exist_ok=True)
RUN=f'{DRIVE_DIR}/{FAMILY}_{time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())}'; SCRATCH=f'/content/d2_scratch'; OUT=f'{RUN}/out'; os.makedirs(OUT)
shutil.rmtree(SCRATCH, ignore_errors=True); subprocess.run(['git','clone','-q',REPO_URL,SCRATCH],check=True); subprocess.run(['git','-C',SCRATCH,'checkout','-q',REPO_COMMIT],check=True)
head=subprocess.check_output(['git','-C',SCRATCH,'rev-parse','HEAD']).decode().strip(); assert head==REPO_COMMIT, head
assert subprocess.check_output(['git','-C',SCRATCH,'status','--porcelain']).decode().strip()=='', 'scratch tree not clean'
MT=SCRATCH; PHASEB=f'{MT}/engine/phaseB'; PHASEC=f'{MT}/phaseC'; INV=f'{PHASEB}/B2_completion_inventory.json'; inv_sha=sha(INV); assert inv_sha==EXPECTED_INVENTORY_SHA256, inv_sha
inv=json.load(open(INV)); PINS=f'{PHASEB}/d/d2_pins.json'; SCRIPT=f'{PHASEB}/d/d2_bankgen.py'
assert sha(PINS)==inv['d_sha256']['d/d2_pins.json'] and sha(SCRIPT)==inv['d_sha256']['d/d2_bankgen.py']
pins=json.load(open(PINS)); assert pins['schema']=='d2_pins_v1' and pins['engine_version']==inv['engine_version'] and sha(f'{PHASEC}/PACKET_INVENTORY.json')==pins['phaseC_inventory_sha256']
lock=dict(launcher_id=LAUNCHER_ID, repo_url=REPO_URL, commit=head, inventory_sha256=inv_sha, pins_sha256=sha(PINS), engine_version=inv['engine_version'], family=FAMILY, run_dir=RUN, reuse_root=REUSE_ROOT or None); json.dump(lock, open(f'{OUT}/launcher_lock.json','w'), indent=1); print(lock)


In [ ]:
# --- 2. registered environment (the script re-verifies live versions and pools as a HARD gate); no CMBtopology needed
ex=pins['environment']; subprocess.run([sys.executable,'-m','pip','install','-q',f"numpy=={ex['numpy']}",f"scipy=={ex['scipy']}",f"healpy=={ex['healpy']}",f"camb=={ex['camb']}",f"pot=={ex['pot']}",'threadpoolctl'],check=True)
os.environ['OPENBLAS_NUM_THREADS']='2'; os.environ['OMP_NUM_THREADS']='2'
import platform, importlib; live={k: importlib.import_module({'pot':'ot'}.get(k,k)).__version__ for k in ('numpy','scipy','healpy','camb','pot')}; live['python']=platform.python_version(); mism={k:(live[k],ex[k]) for k in live if live[k]!=ex[k]}; assert not mism, f'environment lock failed (launcher pre-check): {mism}'
print('environment ok', live)


In [ ]:
# --- 3. generation (long); output goes straight to Drive
DO=f'{OUT}/d2'; args=[sys.executable,SCRIPT,'--mt',MT,'--phaseb',PHASEB,'--phasec',PHASEC,'--out',DO,'--family',FAMILY,'--profile','production_official']+(['--reuse-root',REUSE_ROOT] if REUSE_ROOT else [])
rc=subprocess.run(args,capture_output=True,text=True); open(f'{OUT}/launcher_script_stdout.txt','w').write(rc.stdout); open(f'{OUT}/launcher_script_stderr.txt','w').write(rc.stderr); print(rc.stdout[-3000:])
rm=json.load(open(f'{DO}/d2_run_manifest.json')); script_ok=bool(rc.returncode==0 and rm.get('D2_PASS') is True); print('script rc', rc.returncode, 'D2_PASS', rm.get('D2_PASS'), rm.get('failures'))


In [ ]:
# --- 4. final record (registry + manifests + inventory of the Drive output; the bank arrays stay on Drive) + small zip for the audit
def inventory(root, exclude=(), skip_ext=('.npz',)):
    out={}
    for d,_,fs in os.walk(root):
        for f in fs:
            p=os.path.join(d,f); rel=os.path.relpath(p, root)
            if rel in exclude: continue
            out[rel]=dict(sha256=(sha(p) if not f.endswith(skip_ext) else None), bytes=os.path.getsize(p))
    return out
final=dict(launcher=lock, D2_PASS=bool(script_ok), family=FAMILY, stages=dict(checkout=True, script_returncode=rc.returncode, script_pass=rm.get('D2_PASS'), script_gates=rm.get('gates'), script_failures=rm.get('failures'), directories=rm.get('directories'), timings=rm.get('timings'), seconds=rm.get('seconds')))
final['output_inventory']=inventory(OUT, exclude=('d2_final_record.json',)); json.dump(final, open(f'{OUT}/d2_final_record.json','w'), indent=1)
print(json.dumps({k:final[k] for k in ('D2_PASS','family')}, indent=1), 'run dir:', RUN)
import zipfile
zp=f'/content/d2_{FAMILY}_{REPO_COMMIT[:12]}_audit.zip'
with zipfile.ZipFile(zp,'w',zipfile.ZIP_DEFLATED) as z:
    for d,_,fs in os.walk(OUT):
        for f in fs:
            if not f.endswith('.npz'): z.write(os.path.join(d,f), os.path.relpath(os.path.join(d,f), OUT))
from google.colab import files; print(zp, os.path.getsize(zp)); files.download(zp)
